In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

In [2]:
df = pd.read_csv("Bengaluru_House_Data.csv")

df.head()

,location,size,total_sqft,bath,price
0,Electronic City Phase II,2 BHK,1056,2.0,39.07
1,Chikka Tirupathi,4 Bedroom,2600,5.0,120.00
2,Uttarahalli,3 BHK,1440,2.0,62.00
3,Lingadheeranahalli,3 BHK,1521,3.0,95.00
4,Kothanur,2 BHK,1200,2.0,51.00


In [3]:
print(df.shape)

print(df.columns)

df.info()

(13320, 5)
Index(['location', 'size', 'total_sqft', 'bath', 'price'], dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 13320 entries, 0 to 13319
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   location    13319 non-null  str    
 1   size        13304 non-null  str    
 2   total_sqft  13320 non-null  str    
 3   bath        13247 non-null  float64
 4   price       13320 non-null  float64
dtypes: float64(2), str(3)
memory usage: 817.4 KB


In [4]:
df.isnull().sum()

location       1
size          16
total_sqft     0
bath          73
price          0
dtype: int64

In [5]:
print("Duplicate Rows:", df.duplicated().sum())

df = df.drop_duplicates()

print(df.shape)

Duplicate Rows: 882
(12438, 5)


In [6]:
def convert_sqft(x):

    try:

        if "-" in str(x):

            a, b = x.split("-")

            return (float(a) + float(b)) / 2

        return float(x)

    except:

        return np.nan

In [7]:
df["total_sqft"] = df["total_sqft"].apply(convert_sqft)

In [8]:
df["bhk"] = df["size"].str.extract("(\d+)")

df["bhk"] = pd.to_numeric(
    df["bhk"],
    errors="coerce"
)

In [9]:
df["bath"] = df["bath"].fillna(
    df["bath"].median()
)

In [10]:
df = df.dropna()

df.isnull().sum()

location      0
size          0
total_sqft    0
bath          0
price         0
bhk           0
dtype: int64

In [11]:
df["price_per_sqft"] = (
    df["price"] * 100000
) / df["total_sqft"]

In [12]:
df = df[
    df["total_sqft"] / df["bhk"] >= 300
]

In [13]:
lower = df["price_per_sqft"].quantile(0.05)

upper = df["price_per_sqft"].quantile(0.90)

df = df[
    (df["price_per_sqft"] >= lower)
    &
    (df["price_per_sqft"] <= upper)
]

In [14]:
search_df = df.copy()

In [15]:
location_count = df["location"].value_counts()

df["location_count"] = df["location"].map(location_count)

In [16]:
location_count = df["location"].value_counts()

df["location_count"] = df["location"].map(location_count)

In [17]:
df = df[
    df["location_count"] >= 10
]

In [18]:
location_details = df.groupby("location").agg(

    houses=("location", "count"),

    avg_sqft_price=("price_per_sqft", "mean"),

    avg_price_lakhs=("price", "mean")

).reset_index()

location_details = location_details.sort_values(
    "avg_sqft_price"
)

location_details.head(10)

,location,houses,avg_sqft_price,avg_price_lakhs
119,Kereguddadahalli,10,3442.655403,34.250000
26,Banashankari Stage V,10,3705.071915,49.418000
124,Kothannur,19,3943.757796,49.702632
56,Doddakallasandra,11,3951.162985,53.530000
176,Sompura,11,3985.520523,48.272727
48,Channasandra,33,4021.518232,57.416970
197,Yelenahalli,12,4026.305111,51.185000
35,Begur Road,66,4100.666288,59.402879
43,Bommasandra,31,4118.472598,46.385806
41,Bisuvanahalli,35,4138.022142,42.685143


In [19]:
df = pd.get_dummies(

    df,

    columns=["location"],

    drop_first=True

)

In [20]:
X = df.drop(

    [

        "price",

        "size",

        "price_per_sqft"

    ],

    axis=1

)

In [21]:
y = df["price"]

In [22]:
X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.10,

    random_state=42

)

In [23]:
model = LinearRegression()

model.fit(
    X_train,
    y_train
)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](202,)","[ 0.07, 6.6 ,-6.53,...,10.03,-7.45,17.67]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](202,)","['total_sqft','bath','bhk',...,'location_Yelahanka New Town', 'location_Yelenahalli','location_Yeshwanthpur']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,-31.12
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,202
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int,201


In [24]:
y_pred = model.predict(
    X_test
)

In [25]:
accuracy = r2_score(

    y_test,

    y_pred

)

print(

    "R² Score:",

    accuracy

)

print(

    "Accuracy:",

    accuracy * 100,

    "%"

)

R² Score: 0.8136436628295292
Accuracy: 81.36436628295291 %


In [26]:
mae = mean_absolute_error(

    y_test,

    y_pred

)

rmse = np.sqrt(

    mean_squared_error(

        y_test,

        y_pred

    )

)

print("MAE:", mae)

print("RMSE:", rmse)

MAE: 15.640980086660297
RMSE: 22.824414380408733


In [27]:
loc = input("Enter Location: ")

result = search_df[
    search_df["location"].str.lower()
    ==
    loc.lower()
]

print(result[
    [
        "location",
        "total_sqft",
        "bhk",
        "bath",
        "price",
        "price_per_sqft"
    ]
])

         location  total_sqft  bhk  bath   price  price_per_sqft
5      Whitefield      1170.0  2.0   2.0   38.00     3247.863248
10     Whitefield      1800.0  3.0   2.0   70.00     3888.888889
27     Whitefield      1610.0  3.0   3.0   81.00     5031.055901
47     Whitefield      1459.0  2.0   2.0   94.82     6498.971899
52     Whitefield      2010.0  3.0   3.0   91.00     4527.363184
...           ...         ...  ...   ...     ...             ...
13212  Whitefield       613.0  1.0   1.0   48.00     7830.342577
13235  Whitefield      1730.0  3.0   3.0  125.00     7225.433526
13257  Whitefield      1453.0  3.0   2.0   58.00     3991.741225
13258  Whitefield       877.0  1.0   1.0   59.00     6727.480046
13315  Whitefield      3453.0  5.0   4.0  231.00     6689.834926

[454 rows x 6 columns]


In [28]:
min_sqft = int(input("Minimum Sqft: "))

max_sqft = int(input("Maximum Sqft: "))

In [29]:
range_data = search_df[
    (search_df["total_sqft"] >= min_sqft)
    &
    (search_df["total_sqft"] <= max_sqft)
]

In [30]:
recommend = range_data.groupby("location").agg(

    houses=("location", "count"),

    price_sqft=("price_per_sqft", "mean"),

    price_lakhs=("price", "mean"),

    bhk=("bhk", "mean"),

    bath=("bath", "mean")

).reset_index()

In [31]:
recommend = recommend.sort_values(
    "price_sqft"
)

recommend.head(10)

,location,houses,price_sqft,price_lakhs,bhk,bath


In [32]:
top_locations = search_df.groupby("location").agg(

    total_houses=("location", "count"),

    avg_price_sqft=("price_per_sqft", "mean"),

    avg_price_lakhs=("price", "mean")

).reset_index()

top_locations = top_locations.sort_values(
    "avg_price_sqft"
)

print(top_locations.head(10))

                     location  total_houses  avg_price_sqft  avg_price_lakhs
501                K G Colony             1     3125.000000            50.00
291       Devara Jeevanahalli             1     3151.515152            52.00
674       Maruthi HBCS Layout             1     3160.000000            39.50
115         Asthagrama Layout             1     3166.666667            38.00
952      Thirumalashettyhally             1     3199.628598            34.46
566   Kenchanehalli R R Nagar             1     3200.000000            45.12
871             Shakthi Nagar             1     3200.000000            36.00
799     Raja Rajashweri Nagar             1     3200.000000            45.12
807      Rajarajesheari nagar             1     3200.000000            44.80
1034           Weavers Colony             1     3201.970443            26.00


In [33]:
best = top_locations.iloc[0]

print("Best Location")

print("Location:", best["location"])

print("Price/Sqft:", best["avg_price_sqft"])

print("Average Price:", best["avg_price_lakhs"])

print("Available Houses:", best["total_houses"])

Best Location
Location: K G Colony
Price/Sqft: 3125.0
Average Price: 50.0
Available Houses: 1
